# 01 - Audit dữ liệu và EDA trên tập phát triển

Mục tiêu: xác minh CSV và manifest phép chia bằng `prepare_data`, sau đó mô tả **chỉ tập phát triển (dev)**. Không xem giá trị hay vẽ hình tập test. Dữ liệu gốc được giữ nguyên.

Nguồn: Water Potability của Aditya Kadiwal, Kaggle: https://www.kaggle.com/datasets/adityakadiwal/water-potability. Nguồn tải/checksum thực tế nằm trong `data/source.json` và `data/manifest.json`; không giả định số mẫu theo mô tả bên ngoài.

Giới hạn: dữ liệu không phải mẫu nhóm tự thu thập tại Việt Nam. EDA không chứng minh nhân quả, độc lập có điều kiện hay tiêu chuẩn an toàn. Demo học thuật, không thay thế kiểm nghiệm chất lượng nước. Mọi thống kê được tính bởi mã tự cài trong `src`; pandas chỉ trình bày.

In [ ]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'src' / 'config.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Không tìm thấy project root chứa src/config.py từ cwd/parents.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib
matplotlib.use('Agg')
import pandas as pd
from IPython.display import display, Markdown, Image
from src.config import RAW_DATA_PATH, FEATURE_COLUMNS
from src.train import prepare_data
from src.eda import summarize_features
from src.visualization import create_eda_figures
from src.pipeline import sha256_file
print('Project:', PROJECT_ROOT)

## Kiểm tra dữ liệu và manifest
`prepare_data` kiểm tra cấu trúc/nhãn, loại trùng cùng nhãn, chặn nhóm xung đột trước split và kiểm tra checksum CSV/phép chia. Nếu chưa có manifest, hàm tạo manifest và danh sách split; nếu đã có mà không khớp thì dừng. Audit toàn file ở bước này phục vụ chất lượng dữ liệu, không dùng thống kê test để chọn cấu hình.

In [ ]:
prepared = None
X_dev, y_dev = [], []
if not RAW_DATA_PATH.is_file():
    display(Markdown('**Chưa có CSV gốc.** Chưa chạy audit/EDA; không tạo kết quả minh họa thay dữ liệu thật.'))
else:
    prepared = prepare_data(RAW_DATA_PATH)
    stored_manifest = json.loads(prepared['manifest_path'].read_text(encoding='utf-8'))
    assert stored_manifest['csv_sha256'] == sha256_file(RAW_DATA_PATH)
    assert stored_manifest['split_checksums'] == prepared['manifest']['split_checksums']
    X_dev = [prepared['clean']['X'][index] for index in prepared['dev']]
    y_dev = [prepared['clean']['y'][index] for index in prepared['dev']]
    print('Manifest đã xác minh:', prepared['manifest_path'])
    print('CSV SHA-256:', stored_manifest['csv_sha256'])
    print('Nguồn thực tế:', json.dumps(stored_manifest['source'], ensure_ascii=False, indent=2))
    print('Số hàng gốc / sau làm sạch:', stored_manifest['raw_count'], stored_manifest['clean_count'])
    print('Số mẫu dev dùng cho EDA:', len(X_dev))
    print('Không truy cập giá trị tập test trong notebook này.')

## Thống kê và hình trên dev
Bảng dưới lấy trực tiếp từ `summarize_features`; không dùng pandas aggregation. Hình dùng histogram/boxplot/Pearson do `src` tự tính, giữ điểm cực đoan, không cắt theo ngưỡng an toàn suy đoán. Các file ở `results/notebook_eda` là **EDA tập dev**, không phải kết quả đánh giá test hoặc artifact của một run huấn luyện. Chạy lại sẽ ghi đè các hình cùng tên trong thư mục này.

In [ ]:
if prepared is None:
    print('Bỏ qua EDA: chưa có dữ liệu đã kiểm tra.')
else:
    dev_summary = summarize_features(X_dev, FEATURE_COLUMNS)
    display(Markdown('### Bảng mô tả tập phát triển (dev)'))
    display(pd.DataFrame.from_dict(dev_summary, orient='index'))
    eda_directory = PROJECT_ROOT / 'results' / 'notebook_eda'
    figure_paths = create_eda_figures(X_dev, y_dev, eda_directory)
    print('Đã tạo', len(figure_paths), 'hình dev và eda_manifest.json tại', eda_directory)
    for figure_path in figure_paths:
        display(Markdown('**Tập phát triển (dev): ' + figure_path.name + '**'))
        display(Image(filename=str(figure_path)))